# Benchmark de los modelos base

Comparación de MobileNetV2, ResNet-50 y VGG-16 con las funciones de `src/benchmark.py` y
`src/stats.py`. Las observaciones pareadas son las corridas con la misma semilla y el mismo pliegue
de validación; para cada par (a, b), en el orden de `benchmark.models`, la diferencia es d = a - b.
Las comparaciones por pares usan las corridas que tienen los dos modelos del par, y las conjuntas,
las que tienen todos. Todos los parámetros (métrica primaria, partición, δ, α, potencia, bootstrap
y corrección por comparaciones múltiples) se leen de la sección `benchmark` de `configs/base.yaml`.
Si faltan corridas, cada sección muestra un aviso y el notebook continúa.

## 1. Configuración

Carga de la configuración y formato de las tablas: los números se muestran con seis cifras
significativas, de modo que los p-valores pequeños aparecen en notación científica en lugar de
redondearse a cero.

In [ ]:
import pandas as pd

from src import benchmark, data, final
from src.report import configure_display, show

configure_display()
cfg = data.load_config()
cfg_b = cfg["benchmark"]
pd.Series({
    "modelos": ", ".join(cfg_b["models"]),
    "métrica primaria": cfg_b["primary_metric"],
    "partición de las pruebas": cfg_b["split"],
    "δ (margen de relevancia)": cfg_b["delta"],
    "α": cfg_b["alpha"],
    "potencia": cfg_b["power"],
    "ajuste de p-valores": cfg_b["p_adjust"],
    "bootstrap": f"{cfg_b['bootstrap']['method']}, {cfg_b['bootstrap']['n_resamples']} remuestreos, "
                 f"semilla {cfg_b['bootstrap']['seed']}",
    "McNemar exacto si b + c <": cfg_b["mcnemar_exact_below"],
}, name="valor").to_frame()

Corridas de la ejecución definitiva (`final.label`) de cada modelo: previstas, terminadas y
pareadas con todos los modelos.

In [ ]:
definitivas = benchmark.load_runs(cfg, final.final_label(cfg))
benchmark.availability_table(definitivas)

## 2. Diferencia mínima detectable a posteriori

Sin estudio piloto (desviaciones D2 y D3, capítulo 7, «Número de corridas»): con las corridas
definitivas pareadas, por par de modelos, $\hat{\sigma}_d$ (ddof = 1) de las diferencias de la métrica
primaria en `benchmark.split`, $\delta / \hat{\sigma}_d$ y la diferencia mínima detectable
$\left(t_{1-\alpha/2,\,R-1} + t_{1-\beta,\,R-1}\right) \hat{\sigma}_d \sqrt{1/R + n_{\mathrm{val}}/n_{\mathrm{train}}}$.
Si supera $\delta$, la ausencia de significancia se interpreta como un resultado no concluyente.

In [ ]:
dmd_posteriori = benchmark.posterior_mdd_table(definitivas, cfg)
dmd_posteriori

## 3. Tabla comparativa

Por modelo, sobre `benchmark.split`: cada métrica como media ± desviación estándar entre corridas,
tiempo de entrenamiento por corrida, tiempo de inferencia por imagen en test y parámetros
entrenables (fase B).

In [ ]:
benchmark.comparison_table(definitivas, cfg)

Intervalos de confianza al 95 % de la media de cada métrica: por t de Student (`ic_t_*`) y por
bootstrap (`ic_boot_*`, con el método indicado en `metodo_boot`).

In [ ]:
benchmark.metric_intervals(definitivas, cfg)

Los intervalos anteriores reflejan la variabilidad entre corridas, pero no la de la muestra de
prueba, que es la misma en todas. Intervalo de confianza del 95 % por bootstrap sobre el conjunto
de prueba remuestreando lesiones (`lesion_id`), de modo que las imágenes de una misma lesión entran
o salen juntas: en cada remuestreo se recalcula la exactitud balanceada de cada corrida definitiva
y se promedia entre corridas. Se muestra para la media de cada modelo y para la diferencia entre
cada par (a - b, con los mismos remuestreos para todos los modelos). Remuestreos y semilla en
`benchmark.lesion_bootstrap`.

In [ ]:
show(benchmark.lesion_bootstrap(definitivas, cfg))

La misma tabla comparativa sobre validación.

In [ ]:
benchmark.comparison_table(definitivas, cfg, split="val")

Costo computacional por modelo: media y desviación estándar del tiempo de entrenamiento y de la
inferencia por imagen, y parámetros.

In [ ]:
benchmark.cost_summary(definitivas)

## 4. Ranking y trade-offs

Modelos ordenados por la media de la exactitud balanceada, con su rango medio entre las corridas
pareadas con todos los modelos (k = mejor).

In [ ]:
benchmark.ranking_table(definitivas, cfg)

Exactitud balanceada (media ± desviación estándar) frente a los parámetros entrenables (escala
logarítmica) y frente al tiempo de entrenamiento por corrida.

In [ ]:
show(benchmark.plot_tradeoffs(definitivas, cfg))

## 5. Baseline y margen

### 5.1 Mejor modelo

Mejor y peor modelo por la media de la métrica primaria, con su desviación estándar e IC 95 % por
t de Student y por bootstrap.

In [ ]:
benchmark.best_model_table(definitivas, cfg)

### 5.2 Diferencias entre modelos

Diferencias absolutas y relativas (porcentaje respecto de la media pareada del segundo modelo), con
d de Cohen pareada y r de rango biserial: entre todos los pares, del mejor modelo frente a cada uno
de los demás y de cada uno de los demás frente al peor.

In [ ]:
show({"Todos los pares": benchmark.pairwise_differences(definitivas, cfg),
      "Mejor modelo - otro": benchmark.pairwise_differences(definitivas, cfg, reference="mejor"),
      "Otro - peor modelo": benchmark.pairwise_differences(definitivas, cfg, reference="peor")})

### 5.3 Mejora mínima detectable

Respecto del mejor modelo, con las corridas pareadas disponibles: $\hat{\sigma}_d$, diferencia
mínima detectable, potencia de la t pareada para detectar δ y corridas necesarias para δ.

In [ ]:
benchmark.minimum_detectable_vs_best(definitivas, cfg)

### 5.4 Cotas de referencia

Cotas de `benchmark.reference_bounds` de `base.yaml`, con su valor, métrica, nota y fuente (clave de
`references.bib`). La diferencia entre la media del mejor modelo y la cota solo se calcula para las
cotas medidas con la métrica primaria; las demás quedan vacías porque usan otra métrica.

In [ ]:
benchmark.reference_bounds_table(definitivas, cfg)

## 6. Pruebas de hipótesis

Todas las pruebas usan la métrica primaria sobre `benchmark.split`, con α de `base.yaml`, y
reportan p-valores exactos.

### 6.1 Supuestos

Shapiro-Wilk de las diferencias pareadas de cada par.

In [ ]:
normalidad = benchmark.normality_table(definitivas, cfg)
normalidad

Gráficos Q-Q normales de las diferencias pareadas.

In [ ]:
show(benchmark.plot_qq(definitivas, cfg))

Criterio automático: paramétrico si Shapiro-Wilk no rechaza la normalidad en ningún par; no
paramétrico en otro caso. El criterio determina la prueba principal de 6.2 y 6.3; las dos familias
se reportan siempre.

In [ ]:
benchmark.criterion_table(normalidad)

### 6.2 Comparación conjunta

ANOVA de medidas repetidas (corridas como sujetos, con la corrección de Greenhouse-Geisser y eta
cuadrado parcial) y Friedman (con W de Kendall), sobre las corridas con todos los modelos.

In [ ]:
benchmark.joint_tests(definitivas, cfg)

Post-hoc de Nemenyi a partir de los rangos medios por corrida.

In [ ]:
benchmark.nemenyi_table(definitivas, cfg)

Comparaciones pareadas con p-valores sin ajustar y ajustados por Holm (t pareada y Wilcoxon), y la
prueba principal según el criterio de 6.1.

In [ ]:
pareadas = benchmark.pairwise_tests(definitivas, cfg)
benchmark.pairwise_adjusted_table(pareadas)

### 6.3 Comparaciones pareadas

Por par: diferencia media con IC 95 % por bootstrap, t pareada y Wilcoxon exacto (distribución de
permutación exacta, válida también con empates) con p-valores sin ajustar y ajustados, d de Cohen
pareada, delta de Cliff, probabilidad de superioridad pareada, potencia de la t pareada para δ y
diferencia mínima detectable.

In [ ]:
pareadas

### 6.4 Validación cruzada repetida (Nadeau y Bengio)

t con la varianza corregida $(1/n + n_\text{val}/n_\text{train})\, s^2$, que tiene en cuenta el
solapamiento de los conjuntos de entrenamiento entre corridas, con la razón media entre el tamaño de
validación y el de entrenamiento, en las particiones de `benchmark.nadeau_bengio_splits`.

In [ ]:
benchmark.nadeau_bengio_table(definitivas, cfg)

### 6.5 Equivalencia (TOST)

Dos pruebas t unilaterales pareadas con margen (-δ, δ) para cada par, e IC al 1 - 2α de la
diferencia. Los modelos son equivalentes si `p_tost` < α.

In [ ]:
benchmark.tost_table(definitivas, cfg)

### 6.6 McNemar

McNemar sobre los aciertos por imagen de test de cada par en cada corrida pareada: binomial exacta
si las discordancias b + c son menos de `mcnemar_exact_below`, chi-cuadrado con corrección de
continuidad si no, y p ajustado por Holm entre los pares de cada corrida. Resumen entre corridas.

In [ ]:
mcnemar = benchmark.mcnemar_by_run(definitivas, cfg)
benchmark.mcnemar_summary(mcnemar, cfg)

Resultados por corrida.

In [ ]:
mcnemar

### 6.7 DeLong y AUC macro

DeLong uno contra el resto por clase para cada par en cada corrida pareada (test), con p ajustado
por Holm por separado para cada par y corrida, sobre sus siete contrastes (uno por clase). Resumen
entre corridas por par y clase.

In [ ]:
delong = benchmark.delong_by_run(definitivas, cfg)
benchmark.delong_summary(delong, cfg)

Diferencia de AUC-ROC macro uno contra el resto por corrida pareada, con IC 95 % por bootstrap
percentil estratificado por clase (`bootstrap.n_resamples_auc` remuestreos), y resumen entre corridas.

In [ ]:
auc_bootstrap = benchmark.auc_bootstrap_by_run(definitivas, cfg)
benchmark.auc_bootstrap_summary(auc_bootstrap)

In [ ]:
auc_bootstrap

### 6.8 Descomposición de la varianza

Suma de cuadrados de la métrica primaria atribuible al modelo, a la corrida (separada en semilla,
pliegue y su interacción si las corridas forman un producto completo) y al residual (modelo x
corrida), con su proporción del total.

In [ ]:
benchmark.variance_table(definitivas, cfg)

## 7. Robustez comparada

Métricas de test por fuente (`dataset`) de los tres modelos en una misma tabla: media y desviación
estándar entre corridas.

In [ ]:
benchmark.robustness_by_group(definitivas, cfg, "dataset")